<!-- FULL: keep, DEMO: keep -->
<div style='display:flex; align-items:center; justify-content:space-between; border-bottom: 3px solid rgb(255,106,0); padding-bottom:1em; margin-bottom:1em'>
<div>
<span style='color:rgb(22,60,105); font-size:1.8em; font-weight:bold;'>Introduction to Deep Learning</span><br>
<span style='color:rgb(0,85,100); font-size:1.3em;'>Session 6 &mdash; 4: Optimisers, Schedules &amp; Saving</span><br>
<span style='color:rgb(0,85,100); font-size:1.0em;'>Magda Gregorová &nbsp;·&nbsp; THWS &nbsp;·&nbsp; May 2026</span>
</div>
<img src='../../Common/Pics/thws-logo_vert_en_orange-rgb.png' style='height:80px;'/>
</div>

In [ ]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from torch.utils.data import TensorDataset, DataLoader, random_split
from sklearn.datasets import fetch_california_housing
import numpy as np

<!-- FULL: keep, DEMO: keep -->
<div style='border-left: 4px solid rgb(255,106,0); padding: 0.3em 0.8em; background: rgb(255,250,245); margin: 1.5em 0 0.5em 0;'>
<strong style='color:rgb(22,60,105); font-size:1.05em;'>California Housing dataset</strong>
</div>

<!-- FULL: keep, DEMO: delete -->
A standard regression benchmark — predict median house value from 8 features (income, house age, rooms, etc.) for ~20k California districts.

In [ ]:
housing = fetch_california_housing()
print(f'features: {housing.feature_names}')
print(f'samples: {housing.data.shape[0]}')

In [ ]:
# train/val split before fitting scaler
torch.manual_seed(0)
np.random.seed(0)

n = housing.data.shape[0]
idx = np.random.permutation(n)
n_train = int(0.8 * n)

X_tr_raw = housing.data[idx[:n_train]]
X_va_raw = housing.data[idx[n_train:]]
y_tr_raw = housing.target[idx[:n_train]]
y_va_raw = housing.target[idx[n_train:]]

We normalise inputs using z-score normalisation fitted on training data only (as discussed in Session 5):

$$\tilde{x}_j = \frac{x_j - \mu_j}{\sigma_j}$$

where $\mu_j$ and $\sigma_j$ are the mean and standard deviation of feature $j$ computed on the training set only.

In [ ]:
# fit normalisation statistics on training data only
tr_mean = X_tr_raw.mean(axis=0)
tr_std  = X_tr_raw.std(axis=0)

X_tr = torch.tensor((X_tr_raw - tr_mean) / tr_std, dtype=torch.float32)
X_va = torch.tensor((X_va_raw - tr_mean) / tr_std, dtype=torch.float32)
y_tr = torch.tensor(y_tr_raw, dtype=torch.float32).unsqueeze(1)
y_va = torch.tensor(y_va_raw, dtype=torch.float32).unsqueeze(1)

print(f'X_tr: {X_tr.shape}, y_tr: {y_tr.shape}')
print(f'X_va: {X_va.shape}, y_va: {y_va.shape}')

In [ ]:
train_loader = DataLoader(TensorDataset(X_tr, y_tr), batch_size=64, shuffle=True)
val_loader   = DataLoader(TensorDataset(X_va, y_va), batch_size=64, shuffle=False)
print(f'batches per epoch: {len(train_loader)}')

In [ ]:
class MLP(nn.Module):
    def __init__(self, in_features, hidden_sizes, out_features):
        super().__init__()
        sizes = [in_features] + hidden_sizes
        layers = []
        for i in range(len(sizes) - 1):
            layers.append(nn.Linear(sizes[i], sizes[i+1]))
            layers.append(nn.ReLU())
        layers.append(nn.Linear(sizes[-1], out_features))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

<!-- FULL: keep, DEMO: keep -->
<div style='border-left: 4px solid rgb(255,106,0); padding: 0.3em 0.8em; background: rgb(255,250,245); margin: 1.5em 0 0.5em 0;'>
<strong style='color:rgb(22,60,105); font-size:1.05em;'>torch.optim</strong>
</div>

<!-- FULL: keep, DEMO: delete -->
In notebook 03 we updated parameters manually:

```python
with torch.no_grad():
    for p in model.parameters():
        p -= lr * p.grad
```

`torch.optim` handles this with one line — and adds momentum, adaptive learning rates, and weight decay for free.

The update sequence is always:
1. `optimizer.zero_grad()` — clear accumulated gradients
2. `loss.backward()` — compute gradients
3. `optimizer.step()` — apply update rule


In [ ]:
torch.manual_seed(0)
model = MLP(8, [64, 32], 1)

optimizer = optim.SGD(model.parameters(), lr=0.01)
loss_fn   = nn.MSELoss()

In [ ]:
# one update step
X_batch, y_batch = next(iter(train_loader))

optimizer.zero_grad()                    # 1. clear gradients
loss = loss_fn(model(X_batch), y_batch)  # 2. forward + loss
loss.backward()                          # 3. backward
optimizer.step()                         # 4. update

print(f'loss: {loss.item():.4f}')

<!-- FULL: keep, DEMO: delete -->
<div style='border-left: 4px solid rgb(179,27,27); padding: 0.3em 0.8em; background: rgb(255,240,240); margin: 1.5em 0 0.5em 0;'>
<strong style='color:rgb(179,27,27); font-size:1.05em;'>&#9888; Failure case &mdash; forgetting zero_grad</strong>
</div>

<!-- FULL: keep, DEMO: delete -->
Without `optimizer.zero_grad()`, gradients accumulate across steps — the effective gradient grows each iteration and training diverges.

In [ ]:
torch.manual_seed(0)
model_bad = MLP(8, [64, 32], 1)
opt_bad   = optim.SGD(model_bad.parameters(), lr=0.01)

for step in range(3):
    loss = loss_fn(model_bad(X_batch), y_batch)
    loss.backward()            # no zero_grad!
    opt_bad.step()
    grad_norm = sum(p.grad.norm().item() for p in model_bad.parameters() if p.grad is not None)
    print(f'step {step}: loss={loss.item():.4f}, grad norm={grad_norm:.4f}')

<!-- FULL: keep, DEMO: delete -->
<div style='border-left: 4px solid rgb(179,27,27); padding: 0.3em 0.8em; background: rgb(255,240,240); margin: 1.5em 0 0.5em 0;'>
<strong style='color:rgb(179,27,27); font-size:1.05em;'>&#9888; Failure case &mdash; calling step() before backward()</strong>
</div>

<!-- FULL: keep, DEMO: delete -->
Calling `optimizer.step()` before `loss.backward()` updates parameters using stale gradients from the previous step — or zero gradients on the first step.

In [ ]:
torch.manual_seed(0)
model_bad2 = MLP(8, [64, 32], 1)
opt_bad2   = optim.SGD(model_bad2.parameters(), lr=0.01)

loss = loss_fn(model_bad2(X_batch), y_batch)
opt_bad2.step()     # wrong order — uses stale/zero gradients
loss.backward()
print('parameters updated with wrong gradients')

<!-- FULL: keep, DEMO: delete -->
<div style='border-left: 4px solid rgb(0,85,100); padding: 0.3em 0.8em; background: rgb(235,245,247); margin: 1.5em 0 0.5em 0;'>
<strong style='color:rgb(0,85,100); font-size:1.05em;'>&#8644; Alternatives &mdash; available optimisers</strong>
</div>

<!-- FULL: keep, DEMO: delete -->
All optimisers from Session 5 are available:

```python
optim.SGD(model.parameters(), lr=0.01)
optim.SGD(model.parameters(), lr=0.01, momentum=0.9)
optim.SGD(model.parameters(), lr=0.01, momentum=0.9, weight_decay=1e-4)
optim.Adam(model.parameters(), lr=1e-3)
optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
optim.AdamW(model.parameters(), lr=1e-3)  # Adam with decoupled weight decay
```


<!-- FULL: keep, DEMO: keep -->
<div style='border-left: 4px solid rgb(255,106,0); padding: 0.3em 0.8em; background: rgb(255,250,245); margin: 1.5em 0 0.5em 0;'>
<strong style='color:rgb(22,60,105); font-size:1.05em;'>Clean training functions</strong>
</div>

<!-- FULL: keep, DEMO: delete -->
Separate training and validation logic into reusable functions. This keeps the main loop clean and makes it easy to swap models and optimisers.

In [ ]:
def train_epoch(model, loader, loss_fn, optimizer):
    total = 0.0
    for X_batch, y_batch in loader:
        optimizer.zero_grad()
        loss = loss_fn(model(X_batch), y_batch)
        loss.backward()
        optimizer.step()
        total += loss.item() * len(X_batch)
    return total / len(loader.dataset)


In [ ]:
def val_epoch(model, loader, loss_fn):
    total = 0.0
    with torch.no_grad():
        for X_batch, y_batch in loader:
            loss = loss_fn(model(X_batch), y_batch)
            total += loss.item() * len(X_batch)
    return total / len(loader.dataset)

In [ ]:
def train(model, optimizer, n_epochs=10, seed=42):
    torch.manual_seed(seed)   # fix batch ordering
    loss_fn = nn.MSELoss()
    train_losses, val_losses = [], []
    # # verify identical starting point
    # init_loss = val_epoch(model, val_loader, loss_fn)
    # print(f'initial val loss: {init_loss:.4f}')
    # val_losses.append(init_loss)
    # train_losses.append(init_loss)  # using val_loss as starting point for plotting 
    for epoch in range(n_epochs):
        train_losses.append(train_epoch(model, train_loader, loss_fn, optimizer))
        val_losses.append(val_epoch(model, val_loader, loss_fn))
        print(f'epoch {epoch}: train_loss={train_losses[-1]:4f}, val_loss={val_losses[-1]:4f}')
    return train_losses, val_losses

In [ ]:
def plot_losses(histories, title):
    fig, axes = plt.subplots(1, 2, figsize=(11, 3))
    for label, (tl, vl) in histories.items():
        axes[0].plot(tl, linewidth=2, label=label)
        axes[1].plot(vl, linewidth=2, label=label)
    for ax, split in zip(axes, ['train loss', 'val loss']):
        ax.set_xlabel('epoch')
        ax.set_ylabel('MSE')
        ax.set_title(split, color='#163C69', fontweight='bold')
        ax.legend(fontsize=8)
        ax.spines[['top', 'right']].set_visible(False)
    fig.suptitle(title, color='#163C69', fontweight='bold')
    plt.tight_layout(); plt.show()

<!-- FULL: keep, DEMO: keep -->
<div style='border-left: 4px solid rgb(255,106,0); padding: 0.3em 0.8em; background: rgb(255,250,245); margin: 1.5em 0 0.5em 0;'>
<strong style='color:rgb(22,60,105); font-size:1.05em;'>Optimiser comparison</strong>
</div>

<!-- FULL: keep, DEMO: delete -->
Same architecture and data, different optimisers. The California Housing dataset is large enough that the differences in convergence speed are clearly visible.


In [ ]:
torch.manual_seed(0); m1 = MLP(8, [64, 32], 1)
torch.manual_seed(0); m2 = MLP(8, [64, 32], 1)
torch.manual_seed(0); m3 = MLP(8, [64, 32], 1)

histories = {
    'SGD': train(m1, optim.SGD(m1.parameters(), lr=0.01), n_epochs=10),
    'SGD+momentum': train(m2, optim.SGD(m2.parameters(), lr=0.01, momentum=0.9), n_epochs=10),
    'Adam': train(m3, optim.Adam(m3.parameters(), lr=0.01), n_epochs=10),
}

In [ ]:
plot_losses(histories, 'Optimiser comparison')

<!-- FULL: keep, DEMO: delete -->
As expected from Session 5:

- **SGD** converges slowest — sensitive to learning rate, no momentum
- **SGD + momentum** — faster convergence, smoother curve
- **Adam** — fastest convergence with default hyperparameters

Note that final validation loss matters more than training loss — a faster-converging optimiser is not always better if it overfits more.

**Note:** validation loss can be lower than training loss early in training. This is expected — the training loss is averaged over batches *during* the epoch, when the model is still being updated and is therefore slightly worse than at the end of the epoch. The validation loss is computed *after* all updates for that epoch are complete, so it sees the fully updated model. The concern is not when val loss is close to or below train loss, but when val loss *diverges upward* — that signals overfitting, covered in Session 7.

<!-- FULL: keep, DEMO: keep -->
<div style='border-left: 4px solid rgb(255,106,0); padding: 0.3em 0.8em; background: rgb(255,250,245); margin: 1.5em 0 0.5em 0;'>
<strong style='color:rgb(22,60,105); font-size:1.05em;'>Learning rate schedules</strong>
</div>

<!-- FULL: keep, DEMO: delete -->
A fixed learning rate is a compromise between fast early progress and fine-grained convergence near a minimum. Schedules vary $\eta$ over training to get both.

`torch.optim.lr_scheduler` provides all common schedules. The scheduler's `.step()` is called once per epoch, after the optimiser step.


In [ ]:
torch.manual_seed(0)
model_sched = MLP(8, [64, 32], 1)
optimizer   = optim.SGD(model_sched.parameters(), lr=0.1)

# reduce lr by factor 0.5 every 10 epochs
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)

loss_fn = nn.MSELoss()
train_losses, val_losses, lrs = [], [], []

for epoch in range(30):
    train_losses.append(train_epoch(model_sched, train_loader, loss_fn, optimizer))
    val_losses.append(val_epoch(model_sched, val_loader, loss_fn))
    lrs.append(scheduler.get_last_lr()[0])
    scheduler.step()   # update lr after each epoch


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3))
axes[0].plot(train_losses, color='#005564', linewidth=2, label='train')
axes[0].plot(val_losses,   color='#FF6A00', linewidth=2, label='val', linestyle='--')
axes[0].set_xlabel('epoch'); axes[0].set_ylabel('MSE')
axes[0].set_title('StepLR losses', color='#163C69', fontweight='bold')
axes[0].legend(); axes[0].spines[['top', 'right']].set_visible(False)

axes[1].plot(lrs, color='#163C69', linewidth=2)
axes[1].set_xlabel('epoch'); axes[1].set_ylabel('learning rate')
axes[1].set_title('StepLR schedule', color='#163C69', fontweight='bold')
axes[1].spines[['top', 'right']].set_visible(False)

plt.tight_layout(); plt.show()


<!-- FULL: keep, DEMO: delete -->
<div style='border-left: 4px solid rgb(0,85,100); padding: 0.3em 0.8em; background: rgb(235,245,247); margin: 1.5em 0 0.5em 0;'>
<strong style='color:rgb(0,85,100); font-size:1.05em;'>&#8644; Alternatives &mdash; CosineAnnealingLR</strong>
</div>

<!-- FULL: keep, DEMO: delete -->
Smooth decay following a cosine curve — widely used in modern training:

```python
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=n_epochs, eta_min=1e-4)
```

For Transformers and large-scale training, linear warmup + cosine decay is standard — available via `torch.optim.lr_scheduler.LinearLR` combined with `CosineAnnealingLR` using `SequentialLR`.


<!-- FULL: keep, DEMO: delete -->
<div style='border-left: 4px solid rgb(179,27,27); padding: 0.3em 0.8em; background: rgb(255,240,240); margin: 1.5em 0 0.5em 0;'>
<strong style='color:rgb(179,27,27); font-size:1.05em;'>&#9888; Failure case &mdash; calling scheduler.step() before optimizer.step()</strong>
</div>

<!-- FULL: keep, DEMO: delete -->
In PyTorch ≥ 1.1, `scheduler.step()` must come **after** `optimizer.step()`. Calling it before produces a warning and incorrect learning rate values for the first epoch.


<!-- FULL: keep, DEMO: keep -->
<div style='border-left: 4px solid rgb(255,106,0); padding: 0.3em 0.8em; background: rgb(255,250,245); margin: 1.5em 0 0.5em 0;'>
<strong style='color:rgb(22,60,105); font-size:1.05em;'>Saving and loading models</strong>
</div>

<!-- FULL: keep, DEMO: delete -->
Training a model takes time — always save the result. `state_dict()` returns an `OrderedDict` of all parameter tensors. This is the recommended way to save — it is portable across code changes.


In [ ]:
# inspect state_dict
for name, tensor in m3.state_dict().items():
    print(f'{name}: {list(tensor.shape)}')

In [ ]:
# save
torch.save(m3.state_dict(), 'model.pt')
print('saved model.pt')

In [ ]:
# load into a new model instance
torch.manual_seed(0)
model_loaded = MLP(8, [64, 32], 1)
model_loaded.load_state_dict(torch.load('model.pt'))
print('loaded')

In [ ]:
# verify predictions match
with torch.no_grad():
    y1 = m3(X_va[:4])
    y2 = model_loaded(X_va[:4])
print(f'predictions match: {torch.allclose(y1, y2)}')

<!-- FULL: keep, DEMO: delete -->
<div style='border-left: 4px solid rgb(0,85,100); padding: 0.3em 0.8em; background: rgb(235,245,247); margin: 1.5em 0 0.5em 0;'>
<strong style='color:rgb(0,85,100); font-size:1.05em;'>&#8644; Alternatives &mdash; saving full model vs state_dict</strong>
</div>

<!-- FULL: keep, DEMO: delete -->
You can also save the entire model object:

```python
torch.save(model, 'model_full.pt')          # saves model + architecture
model = torch.load('model_full.pt')         # loads both
```

This is convenient but fragile — it breaks if the class definition changes. `state_dict` is more robust and is the recommended approach.


<!-- FULL: keep, DEMO: keep -->
**Summary of the full training toolkit:**

| Step | PyTorch API |
|---|---|
| Load data | `Dataset`, `DataLoader`, `random_split` |
| Define model | `nn.Module` subclass |
| Define loss | `nn.MSELoss`, `nn.CrossEntropyLoss`, ... |
| Define optimiser | `optim.SGD`, `optim.Adam`, ... |
| Train | `zero_grad → backward → step` |
| Schedule lr | `lr_scheduler.StepLR`, `CosineAnnealingLR` |
| Validate | `torch.no_grad()` |
| Save | `torch.save(model.state_dict(), ...)` |

**Next:** Session 7 — Regularisation &amp; Generalisation
